

# MODULE 34 — SPEECH PROCESSING

# 34.6 Audio Preprocessing

Raw audio recordings are rarely ready to be consumed directly by an ML model.

Imagine building a speech-recognition system that accepts recordings from different mobile phones, microphones and environments.

One recording might be:

- 44,100 Hz, stereo, 30 seconds.
- Another might be 48,000 Hz, mono, 15 seconds.
- Another might contain 5 seconds of silence.
- Another might contain background noise and inconsistent volume.

Our model needs a consistent input representation.

That's exactly what audio preprocessing does.

---

## 1. Understanding the Preprocessing Pipeline

![Audio deep learning and automatic speech recognition pipeline](https://images.openai.com/static-rsc-4/GO1vxWo99Brb9a_f6Ukxj5Ocbx5JP1DMp5s9oZriFTLaZlBPSiV8YdF0au3JzWkGoJ0x3K8_8nwTefHfcFq80DtvuMuH2zIwKay9ZZMaOXehiUmO4LM8sFQa9BO5nEu11t-OvO4TiRUwhZya-lgRiQsFtTkL3b2rrwv87a4lDjY?purpose=inline)

Source: [Audio Deep Learning Made Simple — Automatic Speech Recognition (ASR), How it Works](https://ketanhdoshi.github.io/Audio-ASR/)

```text
Raw Audio
    |
    ▼
Audio Loading
    |
    ▼
Resampling
    |
    ▼
Mono Conversion
    |
    ▼
Normalization
    |
    ▼
Silence Removal / VAD
    |
    ▼
Audio Segmentation
    |
    ▼
Padding / Trimming
    |
    ▼
Audio Augmentation
    |
    ▼
Model-Ready Audio
```

This is a conceptual pipeline. The exact order depends on the model and application.

For example, augmentation is generally applied during training, not to every production inference recording.

---

## 2. Audio Loading

**Purpose:** Read an audio file and convert it into a numerical representation that Python can process.

### Common audio formats

| Format | Characteristics |
|---|---|
| WAV | Common uncompressed audio container |
| MP3 | Lossy compressed audio |
| FLAC | Lossless compressed audio |
| OGG | Audio container commonly using Vorbis or Opus |

When we load audio, we generally obtain two important things:

1. Audio waveform (samples).
2. Sampling rate (samples per second).

### Implementation

```python
import librosa
import numpy as np
import soundfile as sf

audio_path = "speech.wav"

audio, sr = librosa.load(
    audio_path,
    sr=None,
    mono=False
)

print("Audio shape:", audio.shape)
print("Sampling rate:", sr)
print("Duration:", audio.shape[-1] / sr)
```

### Notice

- `sr=None` preserves the original sampling rate.
- `mono=False` preserves the available channels.
- The waveform is represented as numerical samples.

**Important:** `librosa.load()` normally returns floating-point audio. Its default behaviour otherwise resamples to 22,050 Hz and converts to mono.

For this experiment, we're explicitly preserving the original properties.

---

## 3. Resampling

<badge color="success">Must Know</badge>

### Why do we need resampling?

Different audio recordings can have different sampling rates.

For example:

| Audio | Sampling rate |
|---|---:|
| Recording A | 44,100 Hz |
| Recording B | 48,000 Hz |
| Recording C | 16,000 Hz |

But suppose our pretrained speech-recognition model expects 16,000 Hz.

We need to convert the recordings to the required sampling rate.

This process is called **resampling**.

![Audio resampling illustration](https://images.openai.com/static-rsc-4/ZZpFPaispQ3SrOn8AlIjClJONZlsaONsEkeb5gJ3lxnW4zmhU8fJyR_itiisjGd6tqxVDd4kwU4l1TLnwzbvrOMD-KcwdMKuEFHnXFGUW_BMeXWz8bxS1iP5scEWd9EHzc7wp3qyCcXNkvQEQ1x-2TEapPUHamj_KNS1j4RBEF8?purpose=inline)

Source: [TensorFlow Speech Recognition Competition WAV Preprocessing](https://alphafan.github.io/posts/tf_sr_wav_pp.html)

### Understanding the mathematics

Suppose:

- Original sampling rate = 44,100 Hz.
- Target sampling rate = 16,000 Hz.

The conversion ratio is:

$$
Ratio=\frac{16000}{44100}
$$

The new number of samples is approximately:

$$
N_{new}=N_{old}\times\frac{f_{new}}{f_{old}}
$$

For a 1-second recording:

- Original: 44,100 samples.
- Resampled: 16,000 samples.

**Important:** Resampling is not simply deleting samples. Proper resampling involves interpolation and anti-aliasing filtering to avoid introducing unwanted frequency components.

### Implementation

```python
target_sr = 16000

resampled_audio = librosa.resample(
    audio,
    orig_sr=sr,
    target_sr=target_sr
)

sr = target_sr

print("New sampling rate:", sr)
print("New duration:", len(resampled_audio) / sr)
```

If your audio is stereo, handle the channel dimension correctly or convert to mono before this example.

**Industry example:** Many pretrained ASR models expect 16 kHz audio. Always check the model's specific input requirements instead of assuming every speech model uses 16 kHz.

---

## 4. Mono Conversion

Audio can contain one or multiple channels.

![Mono audio representation](https://images.openai.com/static-rsc-4/6l3CPPIbTNPesIXDyiB6qr0iNYkF8FgnqdSovXKhMFV2IXJQ3_wNMlYixcE_ZsqWUg_ehnfyai8iYmNNUp7XMUpXVM5rrOZxv4Kf4P90Jy2PtfTvY0JRhceFESS551FfsWi09K7zYKqQmB-Ej_rlSvT62IYAKMJc9Ea9K5SHE10?purpose=inline)

### Mono

One audio channel.

![Stereo audio representation](https://images.openai.com/static-rsc-4/0Nh9DOYvk79NDI5WoTsU0OoQfHyMen5AGXQhuVpS2uoy88aXWEV9VsJGWBZ6cEjoB-7ERmfXQklZblwPyIOcgTPxgZ0nv4qzstSgRsXk_wdrm6q4vq5eZAyvJAMeKpwImX8dgM9buDSV5l-UYkLnHBQ36KNRo2vOgvI_dA5ZfiQ?purpose=inline)

### Stereo

Two audio channels.

### Why convert stereo to mono?

Some speech-recognition models expect a single-channel waveform.

For two channels:

$$
x_{mono}[n]=\frac{x_L[n]+x_R[n]}{2}
$$

Where:

- $x_L$ = Left channel.
- $x_R$ = Right channel.

### Implementation

```python
audio, sr = librosa.load(
    audio_path,
    sr=None,
    mono=False
)

if audio.ndim == 2:
    audio_mono = np.mean(
        audio,
        axis=0
    )
else:
    audio_mono = audio
```

**Important:** Averaging stereo channels is not always ideal. If the channels contain phase differences, averaging can cause cancellation. For speech applications, selecting the cleaner microphone channel may be better.

---

## 5. Audio Normalization

We already studied normalization in the previous module. Here, let's connect it to preprocessing.

Suppose one person speaks very quietly while another speaks loudly.

Normalization can help make their waveform amplitudes more consistent.

### Peak normalization

$$
y[n]=\frac{x[n]}{\max(|x[n]|)}
$$

### Implementation

```python
def normalize_audio(audio):

    peak = np.max(np.abs(audio))

    if peak == 0:
        return audio

    return audio / peak


normalized_audio = normalize_audio(audio_mono)
```

### Two important approaches

| Method | Purpose |
|---|---|
| Peak normalization | Adjust maximum amplitude |
| RMS normalization | Adjust average signal energy |

**Industry note:** Peak normalization does not guarantee equal perceived loudness. For voice applications, RMS or loudness-based processing may be more suitable.

---

## 6. Silence Removal

<badge color="success">Must Know</badge>

### Why remove silence?

Consider this recording:

```text
[ 3 seconds silence ] [ Speech ] [ 5 seconds silence ]
```

The recording contains unnecessary regions that do not contribute much to the speech-recognition task.

Removing unnecessary silence can:

- Reduce processing time.
- Reduce unnecessary input length.
- Improve audio segmentation.
- Reduce computational requirements.

![Silence and speech regions in an audio recording](https://images.openai.com/static-rsc-4/IrS4cGnOlgplIoISWM1TjM8yntkVyjzdxtLlJRNU1pD8tXZw-7ANe8DmnVqWMwp91NrM1ojswPLw7mwvUF2wDnj33uK8y4Cpxl_TPhk2Djtvb39BhQZFU2iV3_COBOkLFv8dpOuFnfZ-uyiBNGmcDC4bwtIF0WIiTyIzAL4ROp4?purpose=inline)

Source: [Age Prediction from Korean Speech Data Using Neural Networks with Diverse Voice Features](https://www.mdpi.com/2076-3417/15/3/1337)

### Silence removal using librosa

Librosa provides `effects.trim()` to remove silence from the beginning and end of an audio recording based on an amplitude threshold.

```python
trimmed_audio, trim_indices = librosa.effects.trim(
    normalized_audio,
    top_db=30
)

print(
    "Original duration:",
    len(normalized_audio) / sr
)

print(
    "Trimmed duration:",
    len(trimmed_audio) / sr
)
```

### What is `top_db=30`?

It defines the threshold relative to the maximum signal level in decibels.

For example, portions sufficiently below the reference level are considered silent.

**Important:** This operation primarily trims leading and trailing silence. It does not automatically remove every silent interval between words.

### Internal silence removal

For identifying multiple speech regions, we can use:

```python
intervals = librosa.effects.split(
    normalized_audio,
    top_db=30
)

for start, end in intervals:

    print(
        "Start:", start / sr,
        "End:", end / sr
    )
```

This returns intervals of non-silent audio.

Be careful about removing internal pauses indiscriminately: pauses between words and sentences contain useful timing information.

---

## 7. Voice Activity Detection (VAD)

<badge color="success">Must Know</badge>

Silence removal and VAD are related, but they are not the same.

**Voice Activity Detection** identifies whether a particular audio frame contains speech or non-speech.

For example:

```text
Audio Recording
       |
       ▼
  Divide into Frames
       |
       ▼
  Analyze Each Frame
       |
       ├── Speech
       |
       └── Non-Speech
```

![Voice activity detection example](https://images.openai.com/static-rsc-4/DCciVJ3npsJ5K2VSdfQ_97naeZwfOv-LlMskliO_BH3LCt3T11siYOCA8udOHBvBnztKOfnzUhp_yD4dQC0XBpQ20PXUw59b-dQgwrIzQH2MCrrvJniNAncbqNz65r84Mzt_PAxZDFV1baByvLxZe-dcrRnhRclPnJF7ifw0sO4?purpose=inline)

Source: [Voice Activity Detection — Introduction to Speech Processing](https://speechprocessingbook.aalto.fi/Recognition/Voice_activity_detection.html)

### How does VAD work?

Traditional VAD methods examine features such as:

- Short-time energy.
- Zero-crossing rate.
- Spectral characteristics.
- Signal-to-noise ratio.

Modern VAD systems can use trained neural networks.

### Energy-based VAD

One simple approach is to calculate the energy of short audio frames.

$$
E=\frac{1}{N}\sum_{n=1}^{N}x[n]^2
$$

If energy exceeds a chosen threshold, we classify that frame as active.

### Implementation

```python
def energy_vad(
    audio,
    sr,
    frame_ms=30,
    threshold=0.01
):

    frame_size = int(
        sr * frame_ms / 1000
    )

    speech_frames = []

    for start in range(
        0,
        len(audio),
        frame_size
    ):

        frame = audio[
            start:start + frame_size
        ]

        if len(frame) == 0:
            continue

        energy = np.mean(frame ** 2)

        is_speech = energy > threshold

        speech_frames.append(is_speech)

    return np.array(speech_frames)


speech_frames = energy_vad(
    normalized_audio,
    sr
)

print("Speech frames:", np.sum(speech_frames))
print("Total frames:", len(speech_frames))
```

This is a basic demonstration, not a reliable production VAD. Background noise can have higher energy than quiet speech.

### Practical VAD tools

| Tool | Purpose |
|---|---|
| WebRTC VAD | Lightweight speech/non-speech detection |
| Silero VAD | Neural-network-based voice activity detection |
| pyannote.audio | Advanced audio processing, including speech activity and diarization |

![Voice activity detection documentation illustration](https://images.openai.com/static-rsc-4/OebKRrP8I-fGnoSAw67txl-liSKtzdwykc6SNqueN7U5itWcuiYF2_rjVsSASwCOEYgoZ7Kl8bi3UzwWmUvhWlhQnmTyO2JSISHiXx8kh4AwvjqvBjUWpud2EAaxQPhLHBIh_ohOoXu_pcqDIZiXrsxq9-koxmROx-RqSQ2-dV4?purpose=inline)

Source: [Voice Activity Detection — SpeechBrain Documentation](https://speechbrain.readthedocs.io/en/latest/tutorials/tasks/voice-activity-detection.html)

**Industry application:** Meeting transcription systems use VAD to identify speech regions before passing audio to an ASR model.

---

## 8. Audio Segmentation

Segmentation means dividing a long audio recording into smaller audio sections.

Suppose we have a 10-minute recording.

Instead of processing the entire recording as one input, we can divide it into smaller chunks.

```text
10-Minute Audio
       |
       ▼
   Segmentation
       |
       ├── Chunk 1: 0–30 sec
       |
       ├── Chunk 2: 30–60 sec
       |
       ├── Chunk 3: 60–90 sec
       |
       └── ...
```

### Why do we need segmentation?

- Manage long recordings.
- Process audio in batches.
- Handle model input-length restrictions.
- Reduce memory requirements.
- Enable streaming transcription.

### Implementation

Let's divide an audio recording into 10-second chunks.

```python
def segment_audio(
    audio,
    sr,
    chunk_seconds=10
):

    chunk_size = int(
        sr * chunk_seconds
    )

    chunks = []

    for start in range(
        0,
        len(audio),
        chunk_size
    ):

        chunk = audio[
            start:start + chunk_size
        ]

        chunks.append(chunk)

    return chunks


chunks = segment_audio(
    normalized_audio,
    sr,
    chunk_seconds=10
)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):

    print(
        f"Chunk {i + 1}:",
        len(chunk) / sr,
        "seconds"
    )
```

### Fixed vs overlapping segmentation

![Fixed segmentation illustration](https://images.openai.com/static-rsc-4/TcZ-QRRHltKqlXrlmSR0WoKjV_fCSCQuNZpmuDZ3LWbZK38GggZ_szwvT06ntU1Aop-PBnP-EKAVSZbyXVBp8C2prCfvQZ4ujUvc3HqrKU5N3t3PdAFqnk4-uPq6eZ_xAXgSrsmM4hSgjyLOaNr8Houh3ug5pbZFOuRTuCL8rwM?purpose=inline)

**Fixed segmentation**

Chunks are divided without overlap.

![Overlapping segmentation illustration](https://images.openai.com/static-rsc-4/GSiJHIpQJGvfabwkeMqMQnEuxz68sgH4JwBH_fY7Jd3zcOBncrTukDgfI-7rgthzZH3hH7DhSs-9fHuNk4sUYop4jXX9Q0YTnijl5_HZde8DHdt4zF6kfP8T4pWr1uv9wqSlfjyZDK9O4QyLhA3fBB1rop7QgoGT8PyG8KAbdFg?purpose=inline)

**Overlapping segmentation**

Adjacent chunks share some audio.

For example:

- Chunk 1: 0–10 seconds.
- Chunk 2: 9–19 seconds.
- Chunk 3: 18–28 seconds.

Here, each chunk overlaps the next by one second.

**Important:** Overlap may create duplicated transcriptions. You need a strategy to merge the results.

---

## 9. Fixed-Length Audio

Many neural-network architectures and batching pipelines benefit from audio inputs having a consistent duration.

For example:

```text
Recording A = 3 seconds
Recording B = 5 seconds
Recording C = 8 seconds
```

We may want every input to have a duration of 5 seconds.

This is called **fixed-length audio preparation**.

There are two main operations:

1. Padding.
2. Trimming.

### 9.1 Padding

Padding means adding samples to an audio recording to reach a desired length.

Suppose:

- Original audio = 3 seconds.
- Required audio = 5 seconds.
- Sampling rate = 16,000 Hz.

Required samples:

$$
N=5\times16000=80000
$$

Original samples:

$$
N=3\times16000=48000
$$

Additional samples:

$$
80000-48000=32000
$$

We add 32,000 zero-valued samples.

![Audio zero padding illustration](https://images.openai.com/static-rsc-4/ZA0zZSbTQfDsJQ8NEDvH8ki1pQzZfAHJFgF6Mz1X44tbIjlLqBGERFfvfgwRPXwDyoHhvI3WEU727RBxOdzBf-EQDLQlneAIU-A8_oJBbhqGwLvm2i8J2tVs7h6QpiU-EbVsSliz0eJ-Jb7_nQ-eZ5PlefY7Zk0-tG8KoEQLaPE?purpose=inline)

Source: [MATLAB Zero Padding](https://www.educba.com/matlab-zero-padding/)

### Implementation

```python
def pad_audio(
    audio,
    sr,
    target_seconds
):

    target_length = int(
        sr * target_seconds
    )

    if len(audio) >= target_length:
        return audio

    padding_length = (
        target_length - len(audio)
    )

    padded_audio = np.pad(
        audio,
        (0, padding_length),
        mode="constant"
    )

    return padded_audio


padded_audio = pad_audio(
    normalized_audio,
    sr,
    target_seconds=5
)

print(
    "Duration:",
    len(padded_audio) / sr
)
```

### 9.2 Trimming

Trimming removes excess samples when the recording is longer than the required duration.

![Audio trimming illustration](https://images.openai.com/static-rsc-4/jkF-dqDSZ_pi6Z31sIu7N-Aj2L_bqesT9CFaTGFjPIg_2jKLSGOXU4JFgvNw2XBLv8au_OrXScnLlKi8fsItztIvkyINdgYnvP_WCfmWu33MJoN-XAgEbh7AnDoHlAJh6MykwAEqJyJuz1elYIs0IZxGIO2Z7ug7FsKBzDJzpns?purpose=inline)

Source: [Multi-Label Emotion Recognition of Korean Speech Data Using Deep Fusion Models](https://www.mdpi.com/2076-3417/14/17/7604)

```python
def trim_audio(
    audio,
    sr,
    target_seconds
):

    target_length = int(
        sr * target_seconds
    )

    trimmed_audio = audio[
        :target_length
    ]

    return trimmed_audio


trimmed_audio = trim_audio(
    normalized_audio,
    sr,
    target_seconds=5
)

print(
    "Duration:",
    len(trimmed_audio) / sr
)
```

### Padding vs trimming

| Padding | Trimming |
|---|---|
| Adds samples | Removes samples |
| Used for short audio | Used for long audio |
| Often uses zeros | Usually crops samples |
| Preserves original samples | May remove useful speech |

**Important:** Padding is not always harmless. If the model does not receive an appropriate attention mask or length information, it may interpret padded regions as part of the input.

---

# 10. Audio Augmentation

<badge color="success">Must Know</badge>

Now we reach a very important concept in machine learning.

### What is audio augmentation?

Audio augmentation means artificially modifying existing audio recordings to create additional training examples.

### Why do we need it?

Suppose we have only 1,000 speech recordings.

A model trained on these recordings may struggle when it encounters:

- Different microphone volumes.
- Background traffic.
- Different speaking speeds.
- Pitch variations.
- Different recording environments.

We can create modified versions of the original recordings to improve the model's robustness.

```text
              Original Audio
                   |
           Audio Augmentation
                   |
         ┌─────────┼─────────┐
         |         |         |
      Add Noise  Time Stretch  Pitch Shift
         |         |         |
         ▼         ▼         ▼
      Audio 1   Audio 2   Audio 3
         |         |         |
         └─────────┼─────────┘
                   |
              Training Dataset
                   |
                   ▼
                 ASR Model
```

![Audio augmentation methods diagram](https://images.openai.com/static-rsc-4/u8mIeNjsg0TCx8nb_JOhx5vU_bO-XDCC0PSXcCG_VW9Hldc5Vxf1jMpQt-KvKrMqWH_PyuzZ6EsoMH-uPTQdikxJ2NFwSxHnm9Bur3i_eFXSWADNOEmwCNrVD0yVui4DPTeVy8mZlA9AP9513hjZgSQESk1byrF31WwbIRlMpMo?purpose=inline)

Source: [Data Augmentation Methods Applied on Audio Signal](https://www.researchgate.net/figure/Data-augmentation-methods-applied-on-audio-signal_fig3_359512375)

Let's understand each technique.

## 10.1 Adding Background Noise

We already implemented Gaussian noise in the previous module.

Now let's connect it to augmentation.

We can take a clean recording and add background noise at different SNR levels.

$$
x_{aug}[n]=x[n]+n[n]
$$

For example:

| SNR | Audio condition |
|---:|---|
| 20 dB | Mild noise |
| 10 dB | Moderate noise |
| 5 dB | Stronger noise |
| 0 dB | Equal speech and noise power |

### Implementation

```python
def add_noise_augmentation(
    audio,
    snr_db
):

    signal_power = np.mean(
        audio ** 2
    )

    noise = np.random.randn(
        len(audio)
    )

    noise_power = np.mean(
        noise ** 2
    )

    desired_noise_power = (
        signal_power /
        (10 ** (snr_db / 10))
    )

    noise = noise * np.sqrt(
        desired_noise_power / noise_power
    )

    return audio + noise


augmented_audio = add_noise_augmentation(
    normalized_audio,
    snr_db=10
)
```

In industry, instead of only Gaussian noise, we can mix actual background recordings such as traffic, wind and office noise.

## 10.2 Time Stretching

Time stretching changes the duration of audio without intentionally changing its pitch.

For example:

- Original: 5 seconds.
- Slower: 6 seconds.
- Faster: 4 seconds.

![Time stretching illustration](https://images.openai.com/static-rsc-4/935cvTpksJV5Hiylgwn5u7i5L3qqp-naX3rb-jIFdhMd4gfMLcqoBGjtaqWef4SENC8hhXMZy9DdmiWEVkN5jg59AN4Z0IamsepCyZAjXyrhLPtS0qoJmObh5dy8_6CIexfANPcpoWwv2mcO0cjxgn1k9-nz3aHv69zlSo7rFHs?purpose=inline)

Source: [My Awesome Phase Vocoder](https://www.tmdarwen.com/my-awesome-phase-vocoder.html)

Librosa provides `effects.time_stretch()`.

Its `rate` parameter controls the speed:

| Rate | Effect |
|---:|---|
| 0.5 | Twice the duration |
| 0.8 | Slower |
| 1.0 | Original |
| 1.2 | Faster |
| 1.5 | Shorter duration |

### Implementation

```python
slow_audio = librosa.effects.time_stretch(
    normalized_audio,
    rate=0.8
)

fast_audio = librosa.effects.time_stretch(
    normalized_audio,
    rate=1.2
)
```

Listen:

```python
from IPython.display import Audio, display

display(
    Audio(normalized_audio, rate=sr)
)

display(
    Audio(slow_audio, rate=sr)
)

display(
    Audio(fast_audio, rate=sr)
)
```

**Important:** Time stretching attempts to preserve pitch while changing duration, but extreme rates can introduce artifacts.

## 10.3 Pitch Shifting

Pitch shifting changes the perceived pitch of audio while approximately preserving its duration.

For example:

- Original voice.
- Higher-pitched voice.
- Lower-pitched voice.

![Pitch shifting illustration](https://images.openai.com/static-rsc-4/6dPiUVjWsE3YU3y7LSSNedGPKnuKTI7NFeg_jmoIjCh_nQUnY7KOYA6u1HIyj9MBuRAKJ9RwyOhLpkgl94cGXq5svB9N32xTp6VP8o9TXq4VsF_2G20LjD2B_A_a57Ow1TIIh6OMvgTWC7dOTg2Pn-DLRB3wiTFzIYqxgXNt5cM?purpose=inline)

Source: [PitchShift — Audiomentations Documentation](https://iver56.github.io/audiomentations/waveform_transforms/pitch_shift/)

Pitch is measured in Hertz.

A pitch shift of one semitone changes frequency by a factor of:

$$
2^{1/12}
$$

For $n$ semitones:

$$
f_{new}=f_{original}\times 2^{n/12}
$$

### Implementation

```python
higher_pitch = librosa.effects.pitch_shift(
    normalized_audio,
    sr=sr,
    n_steps=2
)

lower_pitch = librosa.effects.pitch_shift(
    normalized_audio,
    sr=sr,
    n_steps=-2
)
```

Here:

- `n_steps=2` raises pitch by two semitones.
- `n_steps=-2` lowers pitch by two semitones.

**Industry application:** Speech emotion recognition and speaker-robustness experiments.

However, excessive pitch shifting can change speaker identity characteristics and should be used carefully.

## 10.4 Volume Variation

Different microphones and recording distances produce different volume levels.

Volume augmentation randomly changes the amplitude of the audio.

Mathematically:

$$
y[n]=g\times x[n]
$$

Where $g$ is the gain factor.

For example:

- $g=0.5$: Quieter.
- $g=1.0$: Unchanged.
- $g=1.5$: Louder.

### Implementation

```python
def volume_augmentation(
    audio,
    min_gain=0.5,
    max_gain=1.5
):

    gain = np.random.uniform(
        min_gain,
        max_gain
    )

    augmented = audio * gain

    return augmented


volume_audio = volume_augmentation(
    normalized_audio
)
```

Avoid clipping when increasing volume. Do not automatically normalize every augmented sample afterward if doing so would eliminate the intended loudness variation.

## 10.5 Time Shifting

Time shifting moves an audio signal forward or backward along the time axis.

For example:

```text
Original:
[ Speech              ]

Shift Right:
[     Speech          ]

Shift Left:
[Speech               ]
```

![Audio time shifting illustration](https://images.openai.com/static-rsc-4/s_W2UfSysdvg8fh5MYZFdte6CkooneOLgcJyWBhEqH0EUdy_ZRdAFkMz0iBf1XyLGRpevhnxYZkk-zn_Gm3GPOMzX-5Ot7JBe6p84iJjpzHydtLlR1KldhSNIorhxg6LQtaCWqId9q6tkST8k5OwEQuvBA4WddrmYDzlG9rh338?purpose=inline)

Source: [Data Augmentation Techniques for Audio Data in Python](https://medium.com/data-science/data-augmentation-techniques-for-audio-data-in-python-15505483c63c)

### Implementation

```python
def time_shift(
    audio,
    shift_seconds,
    sr
):

    shift_samples = int(
        shift_seconds * sr
    )

    shifted = np.zeros_like(audio)

    if shift_samples > 0:

        shifted[shift_samples:] = audio[
            :-shift_samples
        ]

    elif shift_samples < 0:

        shifted[:shift_samples] = audio[
            -shift_samples:
        ]

    else:

        shifted = audio.copy()

    return shifted


shifted_audio = time_shift(
    normalized_audio,
    shift_seconds=0.3,
    sr=sr
)
```

Here, positive values shift audio right and negative values shift it left.

Unlike `np.roll()`, this implementation does not wrap the end of the recording back to the beginning.

Applications: Making a model robust to different speech starting positions and timing variations.

## 10.6 Speed Variation

This is closely related to time stretching, but there is an important distinction.

| Time stretching | Speed variation |
|---|---|
| Changes duration | Changes playback speed |
| Attempts to preserve pitch | Usually changes pitch |
| Uses time-scale modification | Can use playback-rate modification |

For example, if we play audio recorded at 16 kHz as though it were sampled at 18 kHz, it will play faster and sound higher-pitched.

### Implementation

```python
# Original playback
display(
    Audio(normalized_audio, rate=sr)
)

# Faster playback
display(
    Audio(
        normalized_audio,
        rate=int(sr * 1.2)
    )
)

# Slower playback
display(
    Audio(
        normalized_audio,
        rate=int(sr * 0.8)
    )
)
```

Notice that the second and third examples change both duration and pitch.

This is useful to understand because time stretching and speed perturbation are not interchangeable augmentation techniques.

---

## 11. Audio Augmentation — Comparison

| Technique | What changes? | Pitch preserved? |
|---|---|---|
| Background noise | Noise level | Usually |
| Time stretching | Duration | Approximately |
| Pitch shifting | Pitch | No |
| Volume variation | Amplitude | Yes |
| Time shifting | Starting position | Yes |
| Speed variation | Duration and pitch | No |

### When should we use these techniques?

| Application | Useful augmentation |
|---|---|
| Speech recognition | Background noise, speed perturbation |
| Speech emotion recognition | Noise, pitch and volume variation |
| Voice assistants | Noise, time shifting |
| Audio classification | Noise, time stretching, volume |

**Important ML practice:** Apply augmentation to training data, not validation or test data. Otherwise, your evaluation may no longer represent the real-world input distribution.

Also, augmentation must preserve the task label. For example, aggressive pitch changes might alter emotion-related features.

---

# 12. Practical Lab — Complete Audio Preprocessing Toolkit

Now let's combine everything into one reusable preprocessing workflow.

![Audio deep learning and sound classification](https://images.openai.com/static-rsc-4/aUfmtUfTvMMHfb5HAZ0dl_n5O1dnqYKRgXknYCY6czgfBCXvyIpPByNWO1_-Yhu8p3Ma9s0JmBaozvGrIvwUM2vff60i6QaXVSVmdTsYi4G3_C4pB2c-WNtDuyGWVAfrdPnjOIVlcZ_Lr631ByxB3vyXmtW5m8xkOiwt8Mm56l4?purpose=inline)

Source: [Audio Deep Learning Made Simple — Sound Classification](https://ketanhdoshi.github.io/Audio-Classification/)

### Step 1 — Load and inspect audio

```python
import numpy as np
import librosa
import soundfile as sf
import matplotlib.pyplot as plt

audio, sr = librosa.load(
    "speech.wav",
    sr=None,
    mono=True
)

print("Sampling rate:", sr)
print("Duration:", len(audio) / sr)
print("Samples:", len(audio))
```

### Step 2 — Resample to 16 kHz

```python
target_sr = 16000

if sr != target_sr:

    audio = librosa.resample(
        audio,
        orig_sr=sr,
        target_sr=target_sr
    )

    sr = target_sr
```

### Step 3 — Normalize

```python
peak = np.max(np.abs(audio))

if peak > 0:
    audio = audio / peak
```

### Step 4 — Remove leading and trailing silence

```python
audio, _ = librosa.effects.trim(
    audio,
    top_db=30
)
```

### Step 5 — Divide into chunks

```python
chunks = segment_audio(
    audio,
    sr,
    chunk_seconds=5
)

print("Total chunks:", len(chunks))
```

### Step 6 — Prepare fixed-length chunks

```python
target_duration = 5

processed_chunks = []

for chunk in chunks:

    chunk = pad_audio(
        chunk,
        sr,
        target_duration
    )

    chunk = trim_audio(
        chunk,
        sr,
        target_duration
    )

    processed_chunks.append(chunk)

processed_chunks = np.stack(
    processed_chunks
)

print("Final shape:", processed_chunks.shape)
```

Each chunk now contains exactly 80,000 samples at 16 kHz.

### Step 7 — Apply augmentation during training

```python
augmented_chunks = []

for chunk in processed_chunks:

    choice = np.random.choice([
        "noise",
        "stretch",
        "pitch",
        "volume",
        "original"
    ])

    if choice == "noise":

        result = add_noise_augmentation(
            chunk,
            snr_db=10
        )

    elif choice == "stretch":

        result = librosa.effects.time_stretch(
            chunk,
            rate=0.9
        )

        result = pad_audio(
            result,
            sr,
            5
        )

        result = trim_audio(
            result,
            sr,
            5
        )

    elif choice == "pitch":

        result = librosa.effects.pitch_shift(
            chunk,
            sr=sr,
            n_steps=1
        )
```

This creates an augmented training batch. In a real training pipeline, apply augmentation on the fly and keep validation/test data untouched.

### Step 8 — Save processed audio

```python
sf.write(
    "processed_speech.wav",
    audio,
    sr
)
```

---

## 13. Important Industry Considerations

Before using this pipeline with pretrained ASR models such as Whisper or Wav2Vec 2.0, remember:

1. Always follow the model's expected sampling rate.
2. Preserve speech information when removing silence.
3. Use VAD when identifying speech activity is important.
4. Avoid blindly cropping long recordings.
5. Use padding and attention masks appropriately.
6. Avoid excessive augmentation that changes the intended labels.
7. Maintain consistent preprocessing between training and inference.

One additional distinction: Whisper uses its own audio preprocessing and feature-extraction pipeline. You should not assume that every model needs the same externally normalized, segmented waveform.

---

## What You Should Remember 🧠

1. **Resampling:** Changes the sampling rate using proper signal reconstruction.
2. **Mono conversion:** Converts multiple audio channels into one.
3. **Normalization:** Controls audio amplitude.
4. **Silence removal:** Removes unnecessary silent regions.
5. **VAD:** Identifies speech and non-speech regions.
6. **Segmentation:** Divides long recordings into smaller chunks.
7. **Padding and trimming:** Standardize audio duration.
8. **Augmentation:** Creates varied training examples to improve robustness.

---
